# Rationale generation

Generates rationale-augmented CoNLL-2003 training and validation files.


In [ ]:
%pip install -q openai httpx python-dotenv datasets tqdm


In [ ]:
import os
from pathlib import Path
try:
    from kaggle_secrets import UserSecretsClient
    s = UserSecretsClient()
    for key in [
        "GPT_OSS_BASE_URL",
        "GPT_OSS_API_KEY",
        "GPT_OSS_MODEL",
        "GPT_OSS_HTTP_USER",
        "GPT_OSS_HTTP_PASSWORD",
    ]:
        try:
            os.environ[key] = s.get_secret(key)
        except Exception:
            pass
except ImportError:
    print("kaggle_secrets недоступен — для локального запуска используйте .env")
try:
    from dotenv import load_dotenv
    load_dotenv(Path("/kaggle/working/.env"))
except ImportError:
    pass


## Configuration


In [ ]:
from pathlib import Path

CONLL_DIR = Path("/kaggle/input/datasets/alaakhaled/conll003-englishversion")
WORK_DIR = Path("/kaggle/working")
DOTENV_PATH = Path("/kaggle/working/.env")
DOTENV_PATH = DOTENV_PATH if DOTENV_PATH.is_file() else None

TRAIN_LIMIT = 12_000   # случайная подвыборка из train.txt (~14k в CoNLL-2003)
VAL_LIMIT = 2_000      # случайная подвыборка из valid.txt (~3.2k)
GENERATE_TEST = False  # test.txt не размечаем на этом этапе
TEMPERATURE = 0.2
MAX_TOKENS = 2048
SLEEP_S = 0.0
SEED = 42

print("CONLL_DIR =", CONLL_DIR)
print("WORK_DIR =", WORK_DIR)
print("train limit =", TRAIN_LIMIT, "| val limit =", VAL_LIMIT, "| test =", GENERATE_TEST)
print("MAX_TOKENS =", MAX_TOKENS, "| SEED =", SEED)


## Generation code


In [ ]:
import json
import os
import random
import time
from pathlib import Path
from typing import Any

from tqdm.auto import tqdm

def build_prompt(tokens: list[str], tags: list[str]) -> str:
    words_line = " ".join(tokens)
    tags_line = " ".join(tags)
    return (
        "You assist with named entity recognition (NER) on English CoNLL-style BIO tags "
        "(O, B-PER, I-PER, B-LOC, I-LOC, B-ORG, I-ORG, B-MISC, I-MISC).\n\n"
        "Given the sentence and the gold BIO tags (one tag per word, same order), write "
        "ONE short paragraph in English explaining why these spans and entity types are "
        "reasonable. Do not change or restate the tags as a new sequence. Do not output JSON.\n\n"
        f"Sentence:\n{words_line}\n\n"
        f"Gold BIO tags:\n{tags_line}\n\n"
        "Explanation:"
    )


def make_client():
    import httpx
    from openai import OpenAI

    base_url = os.environ.get("GPT_OSS_BASE_URL", "").strip().rstrip("/")
    api_key = os.environ.get("GPT_OSS_API_KEY", "").strip()
    model = os.environ.get("GPT_OSS_MODEL", "").strip()
    http_user = os.environ.get("GPT_OSS_HTTP_USER", "").strip()
    http_password = os.environ.get("GPT_OSS_HTTP_PASSWORD", "").strip()

    if not base_url or not api_key or not model:
        raise ValueError(
            "Задайте переменные окружения GPT_OSS_BASE_URL, GPT_OSS_API_KEY и GPT_OSS_MODEL "
            "(Kaggle: Secrets или Add-ons → Environment)."
        )

    if not base_url.endswith("/v1"):
        base_url = base_url + "/v1"

    auth = None
    if http_user or http_password:
        auth = httpx.BasicAuth(http_user or "", http_password or "")

    http_client = httpx.Client(auth=auth, timeout=httpx.Timeout(120.0, connect=30.0))
    return OpenAI(api_key=api_key, base_url=base_url, http_client=http_client), model


def extract_rationale_from_message(msg: Any) -> str:
    """
    Текст ответа модели. У моделей вроде gpt-oss ответ может быть в ``reasoning``,
    а ``content`` остаётся null, пока не выделено достаточно max_tokens.
    """
    text = (getattr(msg, "content", None) or "").strip()
    if text:
        return text
    reason = getattr(msg, "reasoning", None)
    if reason is None and hasattr(msg, "model_dump"):
        try:
            reason = msg.model_dump().get("reasoning")
        except Exception:  # noqa: BLE001
            reason = None
    if reason is not None:
        return str(reason).strip()
    return ""


def ner_tags_to_labels(example: dict[str, Any], label_names: list[str]) -> list[str]:
    return [label_names[i] for i in example["ner_tags"]]


def load_conll2003_txt(path: Path) -> list[dict[str, Any]]:
    """
    CoNLL-2003 .txt: предложения разделены пустой строкой; в строке ≥2 полей —
    токен = первый столбец, NER-тег = последний. Строки -DOCSTART- пропускаются.
    """
    path = Path(path)
    cur: list[tuple[str, str]] = []
    out: list[dict[str, Any]] = []

    with path.open(encoding="utf-8", errors="replace") as f:
        for raw in f:
            line = raw.rstrip("\n\r")
            if not line.strip():
                if cur:
                    out.append({"tokens": [a for a, _ in cur], "tags": [b for _, b in cur]})
                    cur = []
                continue
            line_st = line.strip()
            if line_st.startswith("-DOCSTART-"):
                continue
            parts = line_st.split()
            if len(parts) < 2:
                continue
            token, tag = parts[0], parts[-1]
            cur.append((token, tag))
    if cur:
        out.append({"tokens": [a for a, _ in cur], "tags": [b for _, b in cur]})
    return out


def resolve_conll_txt_for_split(conll_dir: Path, split: str) -> Path | None:
    """train.txt / valid.txt|validation.txt|dev.txt / test.txt в каталоге датасета."""
    conll_dir = Path(conll_dir)
    candidates: dict[str, tuple[str, ...]] = {
        "train": ("train.txt", "eng.train", "train.conll"),
        "validation": ("valid.txt", "validation.txt", "dev.txt", "eng.testa"),
        "test": ("test.txt", "eng.testb"),
    }
    for name in candidates.get(split, ()):
        p = conll_dir / name
        if p.is_file():
            return p
    return None


def run_rationale_generation(
    *,
    split: str,
    output_path: Path,
    limit: int = 0,
    temperature: float = 0.2,
    max_tokens: int = 2048,
    sleep_s: float = 0.0,
    seed: int = 42,
    dotenv_path: Path | None = None,
    conll_txt_path: Path | None = None,
) -> Path:
    """Пишет JSONL: id, split, tokens, tags, rationale.

    Если задан conll_txt_path — читает предложения из этого CoNLL-файла.
    Иначе — сплит из Hugging Face ``conll2003``.
    """
    try:
        from dotenv import load_dotenv

        if dotenv_path is not None:
            load_dotenv(dotenv_path)
    except ImportError:
        pass

    random.seed(seed)

    if split not in ("train", "validation", "test"):
        raise ValueError("split must be train|validation|test")

    sents: list[dict[str, Any]] | None = None
    part = None
    label_names: list[str] | None = None

    if conll_txt_path is not None:
        sents = load_conll2003_txt(Path(conll_txt_path))
        n_all = len(sents)
    else:
        from datasets import load_dataset

        ds = load_dataset("conll2003")
        part = ds[split]
        label_names = list(part.features["ner_tags"].feature.names)
        n_all = len(part)

    client, model_name = make_client()
    output_path = Path(output_path)
    output_path.parent.mkdir(parents=True, exist_ok=True)
    tmp_path = output_path.with_suffix(output_path.suffix + ".tmp")

    if limit <= 0:
        indices = list(range(n_all))
        random.shuffle(indices)
    else:
        indices = random.sample(range(n_all), min(limit, n_all))
    n_total = len(indices)

    with tmp_path.open("w", encoding="utf-8") as fout:
        for idx in tqdm(indices, desc="Генерация rationales", unit="ex"):
            if sents is not None:
                ex = sents[idx]
                tokens = ex["tokens"]
                tags = ex["tags"]
            else:
                assert part is not None and label_names is not None
                row = part[idx]
                tokens = row["tokens"]
                tags = ner_tags_to_labels(row, label_names)
            prompt = build_prompt(tokens, tags)

            rationale = ""
            last_err: BaseException | None = None
            for attempt in range(5):
                try:
                    resp = client.chat.completions.create(
                        model=model_name,
                        messages=[{"role": "user", "content": prompt}],
                        temperature=temperature,
                        max_tokens=max_tokens,
                    )
                    choice = resp.choices[0].message
                    rationale = extract_rationale_from_message(choice)
                    if not rationale:
                        ch0 = resp.choices[0]
                        fr = getattr(ch0, "finish_reason", None)
                        raise ValueError(f"Пустой ответ модели (finish_reason={fr!r})")
                    break
                except Exception as e:  # noqa: BLE001
                    last_err = e
                    wait = min(60.0, 2.0**attempt)
                    print(f"[warn] idx={idx} attempt {attempt + 1}/5: {e!r}; sleep {wait:.1f}s")
                    time.sleep(wait)

            if not rationale:
                raise RuntimeError(f"Не удалось получить rationale для idx={idx}: {last_err!r}")

            row = {
                "id": int(idx),
                "split": split,
                "tokens": tokens,
                "tags": tags,
                "rationale": rationale,
            }
            fout.write(json.dumps(row, ensure_ascii=False) + "\n")
            fout.flush()

            if sleep_s > 0:
                time.sleep(sleep_s)

    tmp_path.replace(output_path)
    print(f"Готово: {n_total} строк -> {output_path}")
    return output_path


def run_rationale_generation_all_conll_txt(
    *,
    conll_dir: Path,
    output_dir: Path,
    limit: int = 0,
    limits: dict[str, int] | None = None,
    splits: tuple[str, ...] = ("train", "validation", "test"),
    temperature: float = 0.2,
    max_tokens: int = 2048,
    sleep_s: float = 0.0,
    seed: int = 42,
    dotenv_path: Path | None = None,
) -> dict[str, Path]:
    """
    Ищет .txt в ``conll_dir`` и пишет JSONL в ``output_dir``.

    ``limits``: лимит предложений на сплит, напр. ``{"train": 12000, "validation": 2000}``.
    ``0`` — весь сплит. Для сплита без ключа используется общий ``limit``.
    """
    output_dir = Path(output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    out_names: dict[str, str] = {
        "train": "train_with_rationale.jsonl",
        "validation": "valid_with_rationale.jsonl",
        "test": "test_with_rationale.jsonl",
    }
    done: dict[str, Path] = {}
    for split in splits:
        if split not in out_names:
            raise ValueError(f"unknown split {split!r}, expected train|validation|test")
        txt = resolve_conll_txt_for_split(conll_dir, split)
        if txt is None:
            print(f"[skip] нет .txt для split={split} в {conll_dir}")
            continue
        split_limit = limit
        if limits is not None and split in limits:
            split_limit = limits[split]
        outp = output_dir / out_names[split]
        print(f"=== {split}: {txt.name} -> {outp.name} (limit={split_limit or 'all'}) ===")
        run_rationale_generation(
            split=split,
            output_path=outp,
            limit=split_limit,
            temperature=temperature,
            max_tokens=max_tokens,
            sleep_s=sleep_s,
            seed=seed + {"train": 0, "validation": 1, "test": 2}.get(split, 0),
            dotenv_path=dotenv_path,
            conll_txt_path=txt,
        )
        done[split] = outp
    if not done:
        raise FileNotFoundError(f"Не найдено ни одного .txt для splits={splits} в {conll_dir}")
    return done




## Run generation


In [ ]:
_splits: tuple[str, ...] = ("train", "validation")
if GENERATE_TEST:
    _splits = ("train", "validation", "test")

OUTPUTS = run_rationale_generation_all_conll_txt(
    conll_dir=CONLL_DIR,
    output_dir=WORK_DIR,
    limits={"train": TRAIN_LIMIT, "validation": VAL_LIMIT},
    splits=_splits,
    temperature=TEMPERATURE,
    max_tokens=MAX_TOKENS,
    sleep_s=SLEEP_S,
    seed=SEED,
    dotenv_path=DOTENV_PATH,
)
OUTPUTS


## Output check


In [ ]:
for split, p in sorted(OUTPUTS.items()):
    with open(p, encoding="utf-8") as f:
        n = sum(1 for _ in f)
    sz_mib = p.stat().st_size / (1024 * 1024)
    print(f"{split}: строк={n}, {sz_mib:.2f} MiB -> {p.resolve()}")
